## Task 1: Multiple-Task Learning

**Goal:** Solve a regression problem in a multiple-task learning (MTL) scheme.

1. Set up reproducibility and device:

```python
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
```

2. Generate a correlated regression dataset — 8 features built as linear combinations of 8 independent base variables with noise. The target `y` depends on non-linear transformations of those features:

```python
def generate_correlated_regression_data(n_samples=3000, random_state=42):
    rng = np.random.default_rng(random_state)
    z = rng.normal(0, 1, size=(n_samples, 8))
    x1, x2, x3, x4 = z[:,0], z[:,1], z[:,2], z[:,3]
    x5, x6, x7, x8 = z[:,4], z[:,5], z[:,6], z[:,7]

    noise_small = lambda scale=0.05: rng.normal(0, scale, size=n_samples)
    noise_mid   = lambda scale=0.15: rng.normal(0, scale, size=n_samples)

    X = np.column_stack([
        x1 + x2 + noise_small(),              # 1
        2.0 * x3 - 0.5 * x4 + noise_small(),  # 2
        -1.2 * x5 + noise_small(),             # 3
        x6 + x7 + noise_small(),               # 4
        0.7 * x7 - 0.7 * x8 + noise_small(),  # 5
        1.5 * x1 - 0.8 * x3 + noise_mid(),    # 6
        x2 + x4 + x6 + noise_mid(),            # 7
        0.5 * x5 + 0.5 * x8 + noise_mid(),    # 8
    ])

    U = np.column_stack([
        X[:,0] + np.sin(X[:,1]),
        X[:,2]*X[:,3] + X[:,4]*X[:,5],
        -X[:,7]*np.sin(X[:,6]),
    ])

    y = (
        3.5 * U[:, 0]
        + 0.8 * (U[:, 1] ** 2)
        + 1.2 * np.sin(U[:, 2])
        + rng.normal(0, 0.7, size=n_samples)
    )
    return X, y
```

3. Split the data into train / val / test sets. Implement `Dataset` and `DataLoader`.

4. Implement the baseline regression model `BaseRegressionNet`:

```python
BaseRegressionNet(
  (network): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.1, inplace=False)
    (8): Linear(in_features=32, out_features=1, bias=True)
  )
)
```

5. Implement the MTL model `MTLAutoencoderRegressor`. The encoder maps input `X` to a latent vector `z` of dimension 4. The decoder reconstructs `X` from `z`. The regressor predicts `y` from `z`. The model returns `(X_hat, y_hat)`:

```python
MTLAutoencoderRegressor(
  (encoder): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.1, inplace=False)
    (8): Linear(in_features=32, out_features=4, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=4, out_features=32, bias=True)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=32, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.1, inplace=False)
    (8): Linear(in_features=64, out_features=8, bias=True)
  )
  (regressor): Sequential(
    (0): Linear(in_features=4, out_features=16, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=16, out_features=1, bias=True)
  )
)
```

6. Train and evaluate both models. The MTL loss is a weighted sum of reconstruction loss and regression loss:

```python
reconstruction_criterion = nn.MSELoss()
regression_criterion     = nn.MSELoss()

X_hat, y_hat = model(X_batch)
recon_loss   = reconstruction_criterion(X_hat, X_batch)
reg_loss     = regression_criterion(y_hat, y_batch)
total_loss   = alpha_recon * recon_loss + beta_reg * reg_loss
```

Use the following training settings:

```python
BATCH_SIZE   = 128
EPOCHS       = 1000
LR           = 1e-3
WEIGHT_DECAY = 1e-4

ALPHA_RECON  = 0.4   # reconstruction loss weight
BETA_REG     = 1.0   # regression loss weight
```

For datasets generated with different random seeds, compute and compare regression metrics: **MSE**, **MAE**, **R²**.

**Assignment:** Implement, train and compare regression models trained in single-task and multiple-task learning schemes.


In [1]:
import torch
from utils import set_seed, generate_correlated_regression_data

SEED = 42
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

set_seed(SEED)

In [ ]:
from sklearn.model_selection import train_test_split

X, y = generate_correlated_regression_data(random_state=SEED)